# Upload Census 2021 Data to Google Cloud Storage

This notebook creates a Google Cloud Storage bucket and uploads the UK Census 2021 dataset (TS001) to it.

## What This Notebook Does

1. Creates a GCS bucket in your project
2. Uploads 7 census CSV files at different geographic levels
3. Uploads metadata documentation
4. Validates the upload

## Dataset Information

**Census 2021 - TS001**: Number of usual residents in households and communal establishments

Geographic levels included:
- Country (England, Wales)
- Region
- Upper Tier Local Authority (UTLA)
- Lower Tier Local Authority (LTLA)
- Middle Layer Super Output Area (MSOA)
- Lower Layer Super Output Area (LSOA)
- Output Area (OA)

## Prerequisites

- GCP project with billing enabled
- Required IAM role: `roles/storage.admin`
- Census data files in `source_data/census2021-ts001/`

## Time Estimate

5-10 minutes

# Section 1: Configuration & Authentication

First, let's set up our configuration variables and verify authentication.

In [1]:
# Configuration variables

PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           
BUCKET_NAME = f"{PROJECT_ID}-census-data"        

print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print(f"  Region: {REGION}")
print(f"  Bucket Name: {BUCKET_NAME}")

📋 Configuration:
  Project ID: johnswain-1200-20260303091357
  Region: us-central1
  Bucket Name: johnswain-1200-20260303091357-census-data


In [2]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

🔐 Authentication Status:
  ✅ Authenticated successfully
  ✅ Default project: graph-demo-471710
  ⚠️  Note: Using PROJECT_ID (johnswain-1200-20260303091357) instead of default (graph-demo-471710)


In [3]:
# Check required IAM permissions
print("🔐 Checking required IAM permissions...")
print()
print("This notebook requires the following permissions:")
print()
print("   1️⃣  Cloud Storage:")
print("      - roles/storage.admin (to create bucket and upload files)")
print()

try:
    user_email = None
    if hasattr(credentials, '_service_account_email'):
        user_email = credentials._service_account_email
    elif hasattr(credentials, 'service_account_email'):
        user_email = credentials.service_account_email
    
    if user_email:
        print(f"   Your account: {user_email}")
    else:
        print("   Your account: (Unable to detect email from credentials)")
except:
    print("   Your account: (Unable to detect email)")

print()
print("💡 To grant these permissions, run this command in your terminal:")
print()

if user_email:
    print(f"   # Grant Cloud Storage Admin")
    print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
    print(f"     --member='user:{user_email}' \\")
    print(f"     --role='roles/storage.admin'")
else:
    print(f"   # Grant Cloud Storage Admin")
    print(f"   gcloud projects add-iam-policy-binding {PROJECT_ID} \\")
    print(f"     --member='user:YOUR_EMAIL' \\")
    print(f"     --role='roles/storage.admin'")
    print()
    print("   (Replace YOUR_EMAIL with your Google Cloud account email)")

print()
print("⏱️  After granting permissions:")
print("   - Wait 1-2 minutes for IAM propagation")
print("   - Then continue with the rest of the notebook")
print()
print("⚠️  Note: If you don't have these permissions, you'll encounter errors later.")

🔐 Checking required IAM permissions...

This notebook requires the following permissions:

   1️⃣  Cloud Storage:
      - roles/storage.admin (to create bucket and upload files)

   Your account: (Unable to detect email from credentials)

💡 To grant these permissions, run this command in your terminal:

   # Grant Cloud Storage Admin
   gcloud projects add-iam-policy-binding johnswain-1200-20260303091357 \
     --member='user:YOUR_EMAIL' \
     --role='roles/storage.admin'

   (Replace YOUR_EMAIL with your Google Cloud account email)

⏱️  After granting permissions:
   - Wait 1-2 minutes for IAM propagation
   - Then continue with the rest of the notebook

⚠️  Note: If you don't have these permissions, you'll encounter errors later.


In [4]:
# Install required packages
!pip install -q google-cloud-storage

print("✅ Packages installed successfully")


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
✅ Packages installed successfully


In [5]:
# Import required libraries
from google.cloud import storage
from google.api_core import exceptions
import os

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


# Section 2: Create GCS Bucket

Create a Cloud Storage bucket to store the census data.

In [6]:
# Initialize Cloud Storage client
storage_client = storage.Client(project=PROJECT_ID)

print("✅ Storage client initialized")

✅ Storage client initialized


In [7]:
# Create GCS bucket
print(f"🪣 Creating bucket: {BUCKET_NAME}")
print()

try:
    # Check if bucket already exists
    bucket = storage_client.bucket(BUCKET_NAME)
    
    if bucket.exists():
        print(f"⚠️  Bucket '{BUCKET_NAME}' already exists")
        print(f"   Will use existing bucket for uploads")
    else:
        # Create new bucket
        bucket = storage_client.create_bucket(
            BUCKET_NAME,
            location=REGION
        )
        print(f"✅ Bucket created successfully")
        print(f"   Name: {bucket.name}")
        print(f"   Location: {bucket.location}")
        print(f"   Storage Class: {bucket.storage_class}")
    
    print()
    print(f"🔗 View bucket in console:")
    print(f"   https://console.cloud.google.com/storage/browser/{BUCKET_NAME}?project={PROJECT_ID}")
    
except exceptions.Conflict:
    print(f"⚠️  Bucket '{BUCKET_NAME}' already exists (possibly in another project)")
    print(f"   Try using a different bucket name")
    raise
except exceptions.Forbidden as e:
    print(f"❌ Permission denied: {e}")
    print(f"   Make sure you have 'roles/storage.admin' role")
    raise
except Exception as e:
    print(f"❌ Error creating bucket: {type(e).__name__}: {e}")
    raise

🪣 Creating bucket: johnswain-1200-20260303091357-census-data

✅ Bucket created successfully
   Name: johnswain-1200-20260303091357-census-data
   Location: US-CENTRAL1
   Storage Class: STANDARD

🔗 View bucket in console:
   https://console.cloud.google.com/storage/browser/johnswain-1200-20260303091357-census-data?project=johnswain-1200-20260303091357


# Section 3: Upload Census Files

Upload all census CSV files and metadata to the bucket.

In [8]:
# Define source data directory
SOURCE_DIR = "../source_data/census2021-ts001"

# Get absolute path
import os
SOURCE_DIR_ABS = os.path.abspath(SOURCE_DIR)

print(f"📂 Source directory: {SOURCE_DIR_ABS}")

# Verify directory exists
if not os.path.exists(SOURCE_DIR_ABS):
    print(f"❌ Source directory not found: {SOURCE_DIR_ABS}")
    raise FileNotFoundError(f"Directory not found: {SOURCE_DIR_ABS}")

print(f"✅ Source directory exists")

📂 Source directory: /Users/johnswain/Development/gcp-demo-dataplex/source_data/census2021-ts001
✅ Source directory exists


In [9]:
# Upload files to GCS
print("📤 Uploading census files...")
print()

uploaded_files = []
skipped_files = []
total_size = 0

# Walk through directory
for root, dirs, files in os.walk(SOURCE_DIR_ABS):
    for filename in files:
        # Skip .DS_Store and other hidden files
        if filename.startswith('.') or filename == '.DS_Store':
            skipped_files.append(filename)
            continue
        
        # Get full local path
        local_path = os.path.join(root, filename)
        
        # Calculate relative path for GCS blob name
        relative_path = os.path.relpath(local_path, SOURCE_DIR_ABS)
        blob_name = f"census2021-ts001/{relative_path}"
        
        # Get file size
        file_size = os.path.getsize(local_path)
        file_size_mb = file_size / (1024 * 1024)
        
        try:
            # Upload file
            blob = bucket.blob(blob_name)
            blob.upload_from_filename(local_path)
            
            uploaded_files.append({
                'name': filename,
                'blob': blob_name,
                'size': file_size
            })
            total_size += file_size
            
            print(f"✅ {filename:50s} ({file_size_mb:>8.2f} MB) -> {blob_name}")
            
        except Exception as e:
            print(f"❌ Failed to upload {filename}: {e}")
            raise

print()
print(f"📊 Upload Summary:")
print(f"   Files uploaded: {len(uploaded_files)}")
print(f"   Files skipped: {len(skipped_files)} ({', '.join(skipped_files) if skipped_files else 'none'})")
print(f"   Total size: {total_size / (1024 * 1024):.2f} MB")

📤 Uploading census files...

✅ census2021-ts001-utla.csv                          (    0.01 MB) -> census2021-ts001/census2021-ts001-utla.csv
✅ census2021-ts001-lsoa.csv                          (    1.71 MB) -> census2021-ts001/census2021-ts001-lsoa.csv
✅ census2021-ts001-oa.csv                            (    7.40 MB) -> census2021-ts001/census2021-ts001-oa.csv
✅ census2021-ts001-msoa.csv                          (    0.35 MB) -> census2021-ts001/census2021-ts001-msoa.csv
✅ census2021-ts001-ltla.csv                          (    0.02 MB) -> census2021-ts001/census2021-ts001-ltla.csv
✅ census2021-ts001-ctry.csv                          (    0.00 MB) -> census2021-ts001/census2021-ts001-ctry.csv
✅ census2021-ts001-rgn.csv                           (    0.00 MB) -> census2021-ts001/census2021-ts001-rgn.csv
✅ ts001-2021-1.txt                                   (    0.00 MB) -> census2021-ts001/metadata/ts001-2021-1.txt

📊 Upload Summary:
   Files uploaded: 8
   Files skipped: 1 (.DS_Store

# Section 4: Validation

Verify that all files were uploaded successfully.

# Section 6: Create BigQuery Dataset

Load the census data from GCS into BigQuery for analysis.

In [13]:
# Install BigQuery client library
!pip install -q google-cloud-bigquery

print("✅ BigQuery library installed")


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
✅ BigQuery library installed


In [14]:
# Import BigQuery library
from google.cloud import bigquery

# Initialize BigQuery client
bq_client = bigquery.Client(project=PROJECT_ID)

print("✅ BigQuery client initialized")

✅ BigQuery client initialized


In [15]:
# Create BigQuery dataset
DATASET_ID = "census_uk_2021"
DATASET_LOCATION = "US"

print(f"📊 Creating BigQuery dataset: {DATASET_ID}")
print()

try:
    dataset_id = f"{PROJECT_ID}.{DATASET_ID}"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = DATASET_LOCATION
    
    dataset = bq_client.create_dataset(dataset, exists_ok=True)
    
    print(f"✅ Dataset created/verified: {dataset_id}")
    print(f"   Location: {DATASET_LOCATION}")
    print()
    print(f"🔗 View dataset in console:")
    print(f"   https://console.cloud.google.com/bigquery?project={PROJECT_ID}&d={DATASET_ID}")
    
except Exception as e:
    print(f"❌ Error creating dataset: {type(e).__name__}: {e}")
    raise

📊 Creating BigQuery dataset: census_uk_2021

✅ Dataset created/verified: johnswain-1200-20260303091357.census_uk_2021
   Location: US

🔗 View dataset in console:
   https://console.cloud.google.com/bigquery?project=johnswain-1200-20260303091357&d=census_uk_2021


# Section 7: Load CSV from GCS to BigQuery

Load the UTLA (Upper Tier Local Authority) census data from Cloud Storage into a BigQuery table.

In [ ]:
# Define table schema
# Rename columns to be BigQuery-friendly (no spaces, lowercase)
TABLE_ID = "ts001_utla"

schema = [
    bigquery.SchemaField("date", "STRING", mode="REQUIRED", description="Census date (2021)"),
    bigquery.SchemaField("geography", "STRING", mode="REQUIRED", description="Local authority name"),
    bigquery.SchemaField("geography_code", "STRING", mode="REQUIRED", description="ONS geography code"),
    bigquery.SchemaField("total_residents", "INTEGER", mode="NULLABLE", description="Total usual residents"),
    bigquery.SchemaField("household_residents", "INTEGER", mode="NULLABLE", description="Residents in households"),
    bigquery.SchemaField("communal_residents", "INTEGER", mode="NULLABLE", description="Residents in communal establishments"),
]

print("📋 Table Schema:")
print()
for field in schema:
    print(f"   {field.name:25s} {field.field_type:10s} {field.mode:10s} - {field.description}")

In [ ]:
# Load CSV from GCS into BigQuery
table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
gcs_uri = f"gs://{BUCKET_NAME}/census2021-ts001/census2021-ts001-utla.csv"

print(f"📤 Loading data from GCS to BigQuery...")
print(f"   Source: {gcs_uri}")
print(f"   Destination: {table_id}")
print()

try:
    # Configure the load job
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        skip_leading_rows=1,  # Skip header row
        source_format=bigquery.SourceFormat.CSV,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,  # Overwrite if exists
    )
    
    # Start the load job
    load_job = bq_client.load_table_from_uri(
        gcs_uri,
        table_id,
        job_config=job_config
    )
    
    # Wait for the job to complete
    load_job.result()
    
    # Get the destination table
    table = bq_client.get_table(table_id)
    
    print(f"✅ Data loaded successfully!")
    print(f"   Rows loaded: {table.num_rows:,}")
    print(f"   Table size: {table.num_bytes / (1024 * 1024):.2f} MB")
    
except Exception as e:
    print(f"❌ Error loading data: {type(e).__name__}: {e}")
    raise

# Section 8: Validation & Query Examples

Verify the data was loaded correctly and run some sample queries.

In [ ]:
# Query 1: Row count validation
query = f"""
SELECT COUNT(*) as row_count
FROM `{table_id}`
"""

print("📊 Query 1: Row Count Validation")
print()

df = bq_client.query(query).to_dataframe()
print(f"   Total rows: {df['row_count'].iloc[0]:,}")
print(f"   Expected: 175 (Upper Tier Local Authorities in England & Wales)")
print()

In [ ]:
# Query 2: Preview data
query = f"""
SELECT *
FROM `{table_id}`
ORDER BY total_residents DESC
LIMIT 10
"""

print("📊 Query 2: Top 10 Local Authorities by Population")
print()

df = bq_client.query(query).to_dataframe()
print(df.to_string(index=False))

In [ ]:
# Query 3: Aggregate statistics
query = f"""
SELECT 
  COUNT(*) as local_authorities,
  SUM(total_residents) as total_population,
  SUM(household_residents) as household_population,
  SUM(communal_residents) as communal_population,
  ROUND(AVG(total_residents), 0) as avg_population_per_authority,
  MAX(total_residents) as max_population,
  MIN(total_residents) as min_population
FROM `{table_id}`
"""

print("📊 Query 3: England & Wales UTLA Statistics")
print()

df = bq_client.query(query).to_dataframe()

for col in df.columns:
    value = df[col].iloc[0]
    if isinstance(value, (int, float)):
        print(f"   {col:35s}: {value:>15,.0f}")
    else:
        print(f"   {col:35s}: {value}")

print()
print(f"   Household population %: {(df['household_population'].iloc[0] / df['total_population'].iloc[0] * 100):.2f}%")
print(f"   Communal population %:  {(df['communal_population'].iloc[0] / df['total_population'].iloc[0] * 100):.2f}%")

In [ ]:
# Display BigQuery Console link
print("🔗 Explore the data in BigQuery Console:")
print()
print(f"   Dataset: https://console.cloud.google.com/bigquery?project={PROJECT_ID}&d={DATASET_ID}")
print(f"   Table:   https://console.cloud.google.com/bigquery?project={PROJECT_ID}&d={DATASET_ID}&t={TABLE_ID}&page=table")
print()
print("💡 Sample queries to try:")
print()
print("   -- Find local authorities with highest communal residence rates")
print(f"   SELECT geography, total_residents, communal_residents,")
print(f"          ROUND(communal_residents / total_residents * 100, 2) as communal_pct")
print(f"   FROM `{table_id}`")
print(f"   ORDER BY communal_pct DESC")
print(f"   LIMIT 10;")
print()
print("   -- Compare household vs communal living")
print(f"   SELECT ")
print(f"     geography,")
print(f"     household_residents,")
print(f"     communal_residents")
print(f"   FROM `{table_id}`")
print(f"   WHERE geography LIKE '%London%'")
print(f"   ORDER BY total_residents DESC;")

In [10]:
# List all blobs in the bucket with census2021-ts001 prefix
print("🔍 Validating uploads...")
print()

blobs = list(storage_client.list_blobs(BUCKET_NAME, prefix="census2021-ts001/"))

print(f"📁 Files in bucket (prefix: census2021-ts001/):")
print()

total_bucket_size = 0
for blob in blobs:
    size_mb = blob.size / (1024 * 1024)
    total_bucket_size += blob.size
    print(f"   {blob.name:60s} {size_mb:>8.2f} MB")

print()
print(f"📊 Bucket Statistics:")
print(f"   Total files: {len(blobs)}")
print(f"   Total size: {total_bucket_size / (1024 * 1024):.2f} MB")
print()

# Expected file count (7 CSV files + 1 metadata file)
EXPECTED_FILES = 8

if len(blobs) == EXPECTED_FILES:
    print(f"✅ Validation passed: All {EXPECTED_FILES} files uploaded successfully")
elif len(blobs) < EXPECTED_FILES:
    print(f"⚠️  Warning: Expected {EXPECTED_FILES} files, but found {len(blobs)}")
else:
    print(f"⚠️  Note: Found {len(blobs)} files (expected {EXPECTED_FILES})")
    print(f"   Additional files may have been uploaded previously")

🔍 Validating uploads...

📁 Files in bucket (prefix: census2021-ts001/):

   census2021-ts001/census2021-ts001-ctry.csv                       0.00 MB
   census2021-ts001/census2021-ts001-lsoa.csv                       1.71 MB
   census2021-ts001/census2021-ts001-ltla.csv                       0.02 MB
   census2021-ts001/census2021-ts001-msoa.csv                       0.35 MB
   census2021-ts001/census2021-ts001-oa.csv                         7.40 MB
   census2021-ts001/census2021-ts001-rgn.csv                        0.00 MB
   census2021-ts001/census2021-ts001-utla.csv                       0.01 MB
   census2021-ts001/metadata/ts001-2021-1.txt                       0.00 MB

📊 Bucket Statistics:
   Total files: 8
   Total size: 9.49 MB

✅ Validation passed: All 8 files uploaded successfully


In [11]:
# Display expected files for reference
print("📋 Expected census files:")
print()

expected_files = [
    ("census2021-ts001-ctry.csv", "Country level"),
    ("census2021-ts001-rgn.csv", "Region level"),
    ("census2021-ts001-utla.csv", "Upper Tier Local Authority"),
    ("census2021-ts001-ltla.csv", "Lower Tier Local Authority"),
    ("census2021-ts001-msoa.csv", "Middle Layer Super Output Area"),
    ("census2021-ts001-lsoa.csv", "Lower Layer Super Output Area"),
    ("census2021-ts001-oa.csv", "Output Area"),
    ("metadata/ts001-2021-1.txt", "Metadata documentation")
]

for filename, description in expected_files:
    blob_name = f"census2021-ts001/{filename}"
    found = any(blob.name == blob_name for blob in blobs)
    status = "✅" if found else "❌"
    print(f"   {status} {filename:45s} - {description}")

📋 Expected census files:

   ✅ census2021-ts001-ctry.csv                     - Country level
   ✅ census2021-ts001-rgn.csv                      - Region level
   ✅ census2021-ts001-utla.csv                     - Upper Tier Local Authority
   ✅ census2021-ts001-ltla.csv                     - Lower Tier Local Authority
   ✅ census2021-ts001-msoa.csv                     - Middle Layer Super Output Area
   ✅ census2021-ts001-lsoa.csv                     - Lower Layer Super Output Area
   ✅ census2021-ts001-oa.csv                       - Output Area
   ✅ metadata/ts001-2021-1.txt                     - Metadata documentation


# Section 5: Summary & Next Steps

Review what was created and how to use the data.

In [12]:
print("="*80)
print("📦 SETUP COMPLETE")
print("="*80)
print()
print("Resources Created:")
print(f"  🪣 GCS Bucket: {BUCKET_NAME}")
print(f"  📁 Files: {len(blobs)} census files uploaded")
print(f"  💾 GCS Size: {total_bucket_size / (1024 * 1024):.2f} MB")
print()
print(f"  📊 BigQuery Dataset: {DATASET_ID}")
print(f"  📋 BigQuery Table: {TABLE_ID}")
print(f"  📈 Rows: {table.num_rows:,} (UTLA level)")
print()
print("Access Your Data:")
print(f"  🔗 GCS Console: https://console.cloud.google.com/storage/browser/{BUCKET_NAME}?project={PROJECT_ID}")
print(f"  📂 Path prefix: gs://{BUCKET_NAME}/census2021-ts001/")
print()
print(f"  🔗 BigQuery Console: https://console.cloud.google.com/bigquery?project={PROJECT_ID}&d={DATASET_ID}")
print(f"  📊 Table: {PROJECT_ID}.{DATASET_ID}.{TABLE_ID}")
print()
print("Next Steps:")
print("  1. Query data in BigQuery Console")
print("  2. Load additional geographic levels (LTLA, MSOA, LSOA, OA)")
print("  3. Create visualizations with Looker Studio or Data Studio")
print("  4. Register in Dataplex for metadata management")
print("  5. Join with other UK census datasets")
print()
print("Sample gsutil commands:")
print(f"  # List files")
print(f"  gsutil ls gs://{BUCKET_NAME}/census2021-ts001/")
print()
print(f"  # Copy a file locally")
print(f"  gsutil cp gs://{BUCKET_NAME}/census2021-ts001/census2021-ts001-ctry.csv .")
print()
print("Sample bq commands:")
print(f"  # Query from command line")
print(f"  bq query --use_legacy_sql=false \\")
print(f"    'SELECT geography, total_residents FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` LIMIT 10'")
print()
print("Cleanup:")
print("  To delete GCS bucket:")
print(f"  gsutil -m rm -r gs://{BUCKET_NAME}")
print()
print("  To delete BigQuery dataset:")
print(f"  bq rm -r -f -d {PROJECT_ID}:{DATASET_ID}")
print()
print("  Or use the Cloud Console to delete resources manually")

📦 UPLOAD COMPLETE

Resources Created:
  🪣 GCS Bucket: johnswain-1200-20260303091357-census-data
  📁 Files: 8 census files uploaded
  💾 Total Size: 9.49 MB

Access Your Data:
  🔗 GCS Console: https://console.cloud.google.com/storage/browser/johnswain-1200-20260303091357-census-data?project=johnswain-1200-20260303091357
  📂 Path prefix: gs://johnswain-1200-20260303091357-census-data/census2021-ts001/

Next Steps:
  1. View files in the GCS Console
  2. Load data into BigQuery for analysis
  3. Use with Dataflow or other GCP services
  4. Register in Dataplex for metadata management

Sample gsutil commands:
  # List files
  gsutil ls gs://johnswain-1200-20260303091357-census-data/census2021-ts001/

  # Copy a file locally
  gsutil cp gs://johnswain-1200-20260303091357-census-data/census2021-ts001/census2021-ts001-ctry.csv .

  # View file contents
  gsutil cat gs://johnswain-1200-20260303091357-census-data/census2021-ts001/metadata/ts001-2021-1.txt

Cleanup:
  To delete the bucket and all